[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/EMQA/blob/main/EMQA_var_calculation/EMQA_var_calculation.ipynb)

# EMQA_var_calculation
Value at Risk (VaR) with GARCH conditional volatility at 95% and 99% confidence using rolling (expanding) window. Compares GARCH-Normal vs GARCH-t.

**Output:** `oil_var_backtest_99.pdf`, `oil_garch_var_bands.pdf`, `oil_var_backtest_summary.pdf`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'none',
    'axes.facecolor': 'none',
    'savefig.facecolor': 'none',
    'savefig.transparent': True,
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.figsize': (12, 6),
})

COLORS = {
    'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32',
    'orange': '#E67E22', 'purple': '#8E44AD', 'gray': '#808080',
    'cyan': '#00BCD4', 'amber': '#B5853F'
}

def save_fig(fig, name):
    fig.savefig(name, bbox_inches='tight', transparent=True, dpi=300)
    print(f"Saved: {name}")
    if name.endswith('.pdf'):
        png_name = name[:-4] + '.png'
        fig.savefig(png_name, bbox_inches='tight', transparent=True, dpi=300)
        print(f"Saved: {png_name}")


In [2]:
import yfinance as yf

def fetch(ticker, start='2020-01-01', end='2025-12-31'):
    d = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        return d['Close'].squeeze().dropna()
    return d['Close'].dropna()


In [ ]:
from arch import arch_model
from scipy.stats import norm, chi2, t as t_dist

# Fetch Brent prices and returns
brent = fetch('BZ=F', start='2018-01-01')
log_ret = (np.log(brent / brent.shift(1)).dropna()) * 100

# 80/20 train/test split
split = int(len(log_ret) * 0.8)
train_ret = log_ret.iloc[:split]
test_ret = log_ret.iloc[split:]

print(f"Total observations: {len(log_ret)}")
print(f"Training set: {len(train_ret)} obs ({train_ret.index[0].date()} to {train_ret.index[-1].date()})")
print(f"Test set: {len(test_ret)} obs ({test_ret.index[0].date()} to {test_ret.index[-1].date()})")

# Rolling 1-step-ahead VaR with expanding window — GARCH-Normal AND GARCH-t
z_95 = norm.ppf(0.05)
z_99 = norm.ppf(0.01)

# Storage for Normal
var_95_n, var_99_n, sigma_n_list, mu_n_list = [], [], [], []
# Storage for Student-t
var_95_t, var_99_t, sigma_t_list = [], [], []

for i in range(len(test_ret)):
    history = log_ret.iloc[:split + i]

    # --- GARCH-Normal ---
    try:
        am_n = arch_model(history, vol='Garch', p=1, q=1, dist='normal', mean='Constant')
        res_n = am_n.fit(disp='off', show_warning=False)
        fc_n = res_n.forecast(horizon=1)
        sig_n = np.sqrt(fc_n.variance.values[-1, 0])
        mu_n = res_n.params.get('mu', res_n.params.iloc[0])
    except Exception:
        sig_n = history.iloc[-60:].std()
        mu_n = history.mean()

    sigma_n_list.append(sig_n)
    mu_n_list.append(mu_n)
    var_95_n.append(mu_n + z_95 * sig_n)
    var_99_n.append(mu_n + z_99 * sig_n)

    # --- GARCH-t ---
    try:
        am_t = arch_model(history, vol='Garch', p=1, q=1, dist='t', mean='Constant')
        res_t = am_t.fit(disp='off', show_warning=False)
        fc_t = res_t.forecast(horizon=1)
        sig_t = np.sqrt(fc_t.variance.values[-1, 0])
        mu_t = res_t.params.get('mu', res_t.params.iloc[0])
        nu = res_t.params.get('nu', res_t.params.iloc[-1])
        # t-distribution quantiles scaled by sigma
        t_95 = t_dist.ppf(0.05, nu)
        t_99 = t_dist.ppf(0.01, nu)
    except Exception:
        sig_t = sig_n
        mu_t = mu_n
        t_95, t_99 = z_95, z_99

    sigma_t_list.append(sig_t)
    var_95_t.append(mu_t + t_95 * sig_t)
    var_99_t.append(mu_t + t_99 * sig_t)

    if (i + 1) % 50 == 0:
        print(f"  Rolling forecast: {i+1}/{len(test_ret)} done")

# Convert to Series
var_95_n = pd.Series(var_95_n, index=test_ret.index)
var_99_n = pd.Series(var_99_n, index=test_ret.index)
var_95_t = pd.Series(var_95_t, index=test_ret.index)
var_99_t = pd.Series(var_99_t, index=test_ret.index)
sigma_n = pd.Series(sigma_n_list, index=test_ret.index)
sigma_t_ser = pd.Series(sigma_t_list, index=test_ret.index)
mu_n_ser = pd.Series(mu_n_list, index=test_ret.index)

# Violations
viol_95_n = test_ret < var_95_n
viol_99_n = test_ret < var_99_n
viol_95_t = test_ret < var_95_t
viol_99_t = test_ret < var_99_t

n_test = len(test_ret)

def kupiec_pof(n_violations, n_total, alpha):
    p_hat = n_violations / n_total
    if p_hat == 0 or p_hat == 1:
        return np.nan, np.nan
    LR = -2 * (n_violations * np.log(alpha / p_hat) +
               (n_total - n_violations) * np.log((1 - alpha) / (1 - p_hat)))
    p_value = 1 - chi2.cdf(LR, df=1)
    return LR, p_value

print(f"\n{'='*60}")
print(f"  Out-of-Sample VaR Backtest — {n_test} test days")
print(f"{'='*60}")
for label, v95, v99 in [('GARCH-Normal', viol_95_n, viol_99_n),
                         ('GARCH-t',      viol_95_t, viol_99_t)]:
    n95, n99 = int(v95.sum()), int(v99.sum())
    LR99, p99 = kupiec_pof(n99, n_test, 0.01)
    LR95, p95 = kupiec_pof(n95, n_test, 0.05)
    print(f"\n  {label}:")
    print(f"    95% VaR: {n95}/{n_test} = {n95/n_test*100:.2f}%  Kupiec p={p95:.4f}")
    print(f"    99% VaR: {n99}/{n_test} = {n99/n_test*100:.2f}%  Kupiec p={p99:.4f}")


In [ ]:
# Chart 1: VaR backtest — returns with Normal VaR thresholds
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(test_ret.index, test_ret.values, color=COLORS['gray'], linewidth=0.5, alpha=0.7, label='Returns')
ax.plot(var_95_n.index, var_95_n.values, color=COLORS['orange'], linewidth=1, label='VaR 95%')
ax.plot(var_99_n.index, var_99_n.values, color=COLORS['red'], linewidth=1, label='VaR 99%')

viol_99_dates = test_ret.index[viol_99_n]
viol_99_vals = test_ret[viol_99_n]
ax.scatter(viol_99_dates, viol_99_vals, color=COLORS['red'], s=20, zorder=5, label='99% Exceedance')

viol_95_only = viol_95_n & ~viol_99_n
ax.scatter(test_ret.index[viol_95_only], test_ret[viol_95_only],
           color=COLORS['orange'], s=15, zorder=4, label='95% Exceedance')

ax.set_xlabel('Date')
ax.set_ylabel('Returns (%)')
ax.set_title('VaR Backtest — Rolling GARCH(1,1) on Brent Crude (Out-of-Sample)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), frameon=False, ncol=5)

plt.tight_layout()
save_fig(fig, 'oil_var_backtest_99.pdf')
plt.show()


In [ ]:
# Chart 2: GARCH volatility and VaR bands on prices
test_prices = brent.loc[test_ret.index]

# Convert return-space sigma to price-space bands
price_upper_1s = test_prices * (1 + sigma_n.values / 100)
price_lower_1s = test_prices * (1 - sigma_n.values / 100)
price_upper_95 = test_prices * (1 - z_95 * sigma_n.values / 100)
price_lower_95 = test_prices * (1 + z_95 * sigma_n.values / 100)
price_upper_99 = test_prices * (1 - z_99 * sigma_n.values / 100)
price_lower_99 = test_prices * (1 + z_99 * sigma_n.values / 100)

fig, ax = plt.subplots(figsize=(12, 6))

# VaR envelopes (wider = more extreme)
ax.fill_between(test_ret.index, price_lower_99.values, price_upper_99.values,
                color=COLORS['red'], alpha=0.08, label='99% VaR envelope')
ax.fill_between(test_ret.index, price_lower_95.values, price_upper_95.values,
                color=COLORS['orange'], alpha=0.10, label='95% VaR envelope')

# Conditional volatility bands
ax.plot(test_ret.index, price_upper_1s.values, color=COLORS['green'],
        linewidth=0.8, alpha=0.7, label='$\\pm\\sigma_t$')
ax.plot(test_ret.index, price_lower_1s.values, color=COLORS['green'],
        linewidth=0.8, alpha=0.7)

# Actual prices
ax.plot(test_ret.index, test_prices.values, color=COLORS['blue'],
        linewidth=1.2, label='Brent Price')

ax.set_xlabel('Date')
ax.set_ylabel('Price (USD/bbl)')
ax.set_title('GARCH(1,1) Volatility and VaR Bands — Brent Crude (Out-of-Sample)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), frameon=False, ncol=4)

plt.tight_layout()
save_fig(fig, 'oil_garch_var_bands.pdf')
plt.show()


In [ ]:
# Chart 3: Kupiec test model comparison — GARCH-Normal vs GARCH-t
models = ['GARCH-Normal', 'GARCH-t']

results = {}
for label, v95, v99 in [('GARCH-Normal', viol_95_n, viol_99_n),
                         ('GARCH-t',      viol_95_t, viol_99_t)]:
    n95, n99 = int(v95.sum()), int(v99.sum())
    LR99, p99 = kupiec_pof(n99, n_test, 0.01)
    LR95, p95 = kupiec_pof(n95, n_test, 0.05)
    results[label] = {
        'n95': n95, 'n99': n99,
        'pct95': n95/n_test*100, 'pct99': n99/n_test*100,
        'LR95': LR95, 'p95': p95,
        'LR99': LR99, 'p99': p99,
    }

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Violation counts
x = np.arange(2)
width = 0.30
ax1 = axes[0]

for j, conf in enumerate(['95', '99']):
    vals = [results[m][f'n{conf}'] for m in models]
    expected = n_test * (0.05 if conf == '95' else 0.01)
    bars = ax1.bar(x + j * width, vals, width, 
                   color=[COLORS['orange'], COLORS['red']][j],
                   alpha=0.8, label=f'{conf}% VaR violations')
    for bar, val in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

# Expected lines
ax1.axhline(n_test * 0.05, color=COLORS['orange'], ls='--', lw=1, alpha=0.5)
ax1.axhline(n_test * 0.01, color=COLORS['red'], ls='--', lw=1, alpha=0.5)
ax1.text(1.65, n_test * 0.05 + 0.3, f'Expected 95%: {n_test*0.05:.0f}',
         fontsize=8, color=COLORS['orange'])
ax1.text(1.65, n_test * 0.01 + 0.3, f'Expected 99%: {n_test*0.01:.0f}',
         fontsize=8, color=COLORS['red'])

ax1.set_xticks(x + width / 2)
ax1.set_xticklabels(models, fontsize=11)
ax1.set_ylabel('Number of Violations')
ax1.set_title('(A) VaR Violations', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)

# Right panel: Kupiec p-values
ax2 = axes[1]
for j, conf in enumerate(['95', '99']):
    vals = [results[m][f'p{conf}'] for m in models]
    bars = ax2.bar(x + j * width, vals, width,
                   color=[COLORS['orange'], COLORS['red']][j],
                   alpha=0.8, label=f'{conf}% Kupiec p-value')
    for bar, val in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.axhline(0.05, color=COLORS['gray'], ls='--', lw=1.5, alpha=0.7)
ax2.text(1.65, 0.06, 'Reject threshold (5%)', fontsize=8, color=COLORS['gray'])

ax2.set_xticks(x + width / 2)
ax2.set_xticklabels(models, fontsize=11)
ax2.set_ylabel('p-value')
ax2.set_title('(B) Kupiec POF Test', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)

fig.suptitle(f'VaR Model Comparison — Brent Crude ({n_test} OOS days)',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'oil_var_backtest_summary.pdf')
plt.show()
